In [ ]:
import json
import random
from collections import defaultdict
from pathlib import Path
from tqdm import tqdm

REPO_ROOT = Path.cwd()
DATA_DIR = REPO_ROOT / "data"
MIXING_DIR = DATA_DIR / "data_mixing" / "tulu3"
SOURCE_PATH = DATA_DIR / "tulu3-sft_source.json"
FILTERED_SOURCE_PATH = DATA_DIR / "tulu3-sft_source_filtered.json"

## Delete irrelevant domains

In [ ]:
with SOURCE_PATH.open("r", encoding="utf-8") as read_file:
    data = json.load(read_file)

sources_to_delete = [
    "ai2-adapt-dev/tulu_v3.9_aya_100k",
    "ai2-adapt-dev/tulu_hard_coded_repeated_10",
]

In [ ]:
filtered_data = [
    item for item in data
    if item.get("source") not in sources_to_delete
]

print(f"Items after deletion: {len(filtered_data)}")

with FILTERED_SOURCE_PATH.open("w", encoding="utf-8") as f:
    json.dump(filtered_data, f, indent=2)


Items after deletion: 737228


In [ ]:
with FILTERED_SOURCE_PATH.open("r", encoding="utf-8") as read_file:
    filtered_data = json.load(read_file)

In [42]:
# Define the mapping from sources to domains
source_to_domain = {
    # Code Domain
    'ai2-adapt-dev/evol_codealpaca_heval_decontaminated': 'code',
    'ai2-adapt-dev/personahub_code_v2_34999': 'code',

    # Precise_IF Domain
    'ai2-adapt-dev/personahub_ifdata_manual_seed_v3_29980': 'precise_IF',

    # Math Domain
    'ai2-adapt-dev/numinamath_tir_math_decontaminated': 'math',
    'ai2-adapt-dev/personahub_math_v5_regen_149960': 'math',
    'ai2-adapt-dev/tulu_v3.9_open_math_2_gsm8k_50k': 'math',
    'ai2-adapt-dev/tulu_v3.9_personahub_math_interm_algebra_20k': 'math',
    'allenai/tulu-3-sft-personas-math-grade': 'math',

    # General Domain
    'ai2-adapt-dev/no_robots_converted': 'general',
    "ai2-adapt-dev/oasst1_converted": 'general',
    'ai2-adapt-dev/tulu_v3.9_wildchat_100k': 'general',

    # Knowledge_Recall Domain
    'ai2-adapt-dev/flan_v2_converted': 'knowledge_recall',
    'ai2-adapt-dev/tulu_v3.9_sciriff_10k': 'knowledge_recall',
    'ai2-adapt-dev/tulu_v3.9_table_gpt_5k': 'knowledge_recall',

    # Safety Domain
    'ai2-adapt-dev/coconot_converted': 'safety',
    'ai2-adapt-dev/tulu_v3.9_synthetic_finalresp_wildguardmixtrain_decontaminated_50k': 'safety',
    'ai2-adapt-dev/tulu_v3.9_wildjailbreak_decontaminated_50k': 'safety',
}

In [43]:
mapped_data = []
unmapped_sources = set()

for item in filtered_data:
    source = item.get('source')
    domain = source_to_domain.get(source)
    if domain:
        item['domain'] = domain
        mapped_data.append(item)
    else:
        unmapped_sources.add(source)

if unmapped_sources:
    print(f"Unmapped sources found and excluded: {unmapped_sources}")

print("Dataset size after mapping to domains:", len(mapped_data))

# Create a nested dictionary: domain -> source -> list of items
domain_source_dict = defaultdict(lambda: defaultdict(list))

for item in mapped_data:
    domain = item['domain']
    source = item['source']
    domain_source_dict[domain][source].append(item)

Dataset size after mapping to domains: 737228


In [ ]:
val_output_tokens_per_domain = 150000

# Initialize a dictionary to hold validation data
validation_data = defaultdict(list)

# Initialize a list to hold items to be removed from the original dataset
validation_items = []

for domain, sources in tqdm(domain_source_dict.items(), desc="Processing domains"):
    # Get all sources in the domain
    source_list = list(sources.keys())
    num_sources = len(source_list)
    
    if num_sources == 0:
        print(f"No sources found for domain '{domain}'. Skipping.")
        continue

    # Calculate target tokens per source for balanced representation
    target_tokens_per_source = val_output_tokens_per_domain / num_sources

    domain_val_items = []
    
    for source in source_list:
        items = sources[source]
        
        random.shuffle(items)
        
        selected = []
        current_tokens = 0
        
        for item in items:
            if current_tokens >= target_tokens_per_source:
                break
            selected.append(item)
            current_tokens += item['output_len']
        
        domain_val_items.extend(selected)
    
    current_total_tokens = sum(item['output_len'] for item in domain_val_items)
    
    if current_total_tokens < val_output_tokens_per_domain:
        additional_needed = val_output_tokens_per_domain - current_total_tokens
        
        # Collect remaining items not yet selected
        remaining_items = []
        for source in source_list:
            for item in sources[source]:
                if item not in domain_val_items:
                    remaining_items.append(item)
        
        random.shuffle(remaining_items)
        
        for item in remaining_items:
            if current_total_tokens >= val_output_tokens_per_domain:
                break
            domain_val_items.append(item)
            current_total_tokens += item['output_len']
    
    # Add the selected validation items to the dictionary
    validation_data[domain] = domain_val_items
    
    # Add to the list of validation items to be removed later
    validation_items.extend(domain_val_items)
    
    # Print the total tokens for this domain's validation set
    print(f"Domain '{domain}': Selected {len(domain_val_items)} validation items with total output tokens {current_total_tokens}")
    


for domain, val_items in validation_data.items():
    filepath = MIXING_DIR / f"tulu3_{domain}_val.jsonl"

    with filepath.open("w", encoding="utf-8") as f:
        json.dump(val_items, f, indent=2)


Processing domains:  33%|███▎      | 2/6 [00:00<00:00, 16.67it/s]

Domain 'code': Selected 702 validation items with total output tokens 150047
Domain 'safety': Selected 1421 validation items with total output tokens 150146
Domain 'knowledge_recall': Selected 2391 validation items with total output tokens 150159


Processing domains: 100%|██████████| 6/6 [00:00<00:00, 16.97it/s]

Domain 'math': Selected 420 validation items with total output tokens 154437
Domain 'general': Selected 555 validation items with total output tokens 150610
Domain 'precise_IF': Selected 444 validation items with total output tokens 151135


In [45]:
validation_set = set()
for item in validation_items:
    # Create a unique identifier for each item
    # Here, we'll use a tuple of all relevant fields
    identifier = (
        item.get('instruction', ''),
        item.get('input', ''),
        item.get('output', ''),
        item.get('output_len', 0),
        item.get('input_len', 0),
        item.get('source', ''),
    )
    validation_set.add(identifier)

# Now, filter out validation items from the original dataset
updated_structured_data = [
    item for item in mapped_data
    if (
        (
            item.get('instruction', ''),
            item.get('input', ''),
            item.get('output', ''),
            item.get('output_len', 0),
            item.get('input_len', 0),
            item.get('source', ''),
        )
        not in validation_set
    )
]

print("Original dataset size after removing validation items:", len(updated_structured_data))

Original dataset size after removing validation items: 731248


In [ ]:
target_path = MIXING_DIR / "tulu3_target.json"
with target_path.open("w", encoding="utf-8") as f:
    json.dump(updated_structured_data, f, indent=2)

print("Updated original dataset saved")

Updated original dataset save
